# ENTREGA 1 - STAR SCHEMA 
- Tabela Fato 
- Tabelas de dimensões (movies, people, companies e reviews)
- Tabelas-Ponte (movie_genre, movie_person, movie_company)

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold;

### Tabela Fato - gold.fact_movies_performance

In [0]:
from pyspark.sql.functions import col

#leitura das tabelas Silver de métricas e financeiro, além da dimensão de filmes da Gold
df_financeiro = spark.table("silver.tb_financeiro_filmes")
df_metricas = spark.table("silver.tb_metricas_engajamento")
df_dim_movies = spark.table("gold.dim_movies")

#junção das tabelas de negócio usando o id_filme
df_fato_base = df_financeiro.join(
    df_metricas, on="id_filme", how="inner"
)

#junção da dimensão de filmes para injetar a Surrogate Key (sk_movie_id)
df_fato_final = df_fato_base.join(
    df_dim_movies.select("sk_movie_id", "id_filme"), 
    on="id_filme", 
    how="inner"
)

#seleção, conversão e reordenação exata das colunas
df_fact_movies_performance = df_fato_final.select(
    col("sk_movie_id").cast("bigint"),
    #Métricas Financeiras
    col("orcamento_usd").cast("decimal(18,2)"),
    col("receita_usd").cast("decimal(18,2)"),
    col("lucro_usd").cast("decimal(18,2)"),
    col("orcamento_brl").cast("decimal(18,2)"),
    col("receita_brl").cast("decimal(18,2)"),
    col("lucro_brl").cast("decimal(18,2)"),
    #Métricas de Engajamento
    col("popularidade").cast("double"),
    col("nota_media_tmdb").cast("double"),
    col("qtd_votos_tmdb").cast("int"),
    col("nota_media_imdb").cast("double"),
    col("qtd_votos_imdb").cast("int")
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_movies_performance")

display(spark.table("gold.fact_movies_performance").limit(10))

### Dimensões

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_silver_movies = spark.table("silver.tb_info_filmes")

#selecao de metadados e conversao 
df_dim_movies = df_silver_movies.select(
    F.col("id_filme").cast("string"), 
    F.col("titulo").cast("string"), 
    F.col("data_lancamento"),
    F.col("ano_lancamento"),
    F.col("duracao_minutos").cast("int"),
    F.col("idioma_original"), 
    F.col("status_filme"), 
    F.col("sinopse")
).dropDuplicates(["id_filme"])      #garante que nenhum filme se repita (em funcao do id)

#geração da Surrogate Key
janela_sk_movies = Window.orderBy("id_filme")
df_dim_movies = (df_dim_movies
    .withColumn("sk_movie_id", F.row_number().over(janela_sk_movies).cast("bigint"))
)

#ano_lancamento ja vem da Silver, nao precisa recriar

#reordenação das colunas exatamente conforme o dicionário de dados
df_dim_movies = df_dim_movies.select(
    "sk_movie_id", 
    "id_filme", 
    "titulo", 
    "data_lancamento", 
    "ano_lancamento", 
    "duracao_minutos", 
    "idioma_original", 
    "status_filme", 
    "sinopse"
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_movies")

display(spark.table("gold.dim_movies").limit(50))

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#decidi começar com a dimensao genero pois é a tabela mais simples vindo da silver

#leitura da tabela silver generos e criacao da dimensao de genero
df_silver_generos = spark.table("silver.tb_generos")
df_dim_generos = df_silver_generos.select("nome_genero").dropDuplicates()

#criacao da surrogate key (sk), usando a func row_number pra criar um id sequencial para ordenação alfabética
janela_sk= Window.orderBy("nome_genero")
df_dim_generos = df_dim_generos.withColumn("sk_genre_id", F.row_number().over(janela_sk))

#reordenando as colunas colocando a primary key no inicio 
df_dim_generos = df_dim_generos.select("sk_genre_id", "nome_genero")

df_dim_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_genres")

display(spark.table("gold.dim_genres").limit(10))


In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df_silver_entidades = spark.table("silver.tb_pessoas_empresas")

#dimensao pessoas - selecionando pelas tags ator, diretor e roteirista e usando alias para renomear as colunas
df_people = (df_silver_entidades
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .dropDuplicates()
)

#surrogate key da dim_people
janela_sk_people = Window.orderBy("nome_pessoa", "tipo_pessoa")
df_people = (df_people
    .withColumn("sk_person_id", F.row_number().over(janela_sk_people).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_people")


#dimensao empresas (produtoras)
df_companies = (df_silver_entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("nome_entidade").alias("nome_empresa"),
        F.col("tipo_entidade").alias("tipo_empresa")
    )
    .dropDuplicates()
)

#surrogate key da dim_companies
janela_sk_companies = Window.orderBy("nome_empresa", "tipo_empresa")
df_companies = (df_companies
    .withColumn("sk_company_id", F.row_number().over(janela_sk_companies).cast("bigint"))
    .select("sk_company_id", "nome_empresa", "tipo_empresa")
)

df_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_companies")


display(spark.table("gold.dim_people").limit(100))
display(spark.table("gold.dim_companies").limit(50))


In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#leitura da tabela silver + tabela gold para usarmos a surrogate key da dimensao movies
df_silver_reviews = spark.table("silver.tb_avaliacoes_usuarios")
df_dim_movies = spark.table("gold.dim_movies")

#filtro de registros com nota_usuario NULL: usuarios que deixaram comentario mas sem nota
#essas linhas nao devem ser agregadas pois nao representam avaliacoes reais
#nota_usuario = 0 é valido (nota minima na escala 0-10), diferente de duracao_minutos/orcamento/receita
#sem este filtro, filmes com apenas notas NULL geram registros com qtd=0 e nota_media=NULL na dim_reviews
df_silver_reviews = df_silver_reviews.filter(F.col("nota_usuario").isNotNull())

#agregação de avaliações por id_filme (calculando contagem e média das notas)
df_avaliacoes = (df_silver_reviews
    .groupBy("id_filme")
    .agg(
        F.count("nota_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

#join com a dimensão de filmes para substituir o id_filme pela sk_movie_id
df_dim_reviews = (df_avaliacoes
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
)

#surrogate key da dim_reviews
janela_sk_reviews = Window.orderBy("sk_movie_id")
df_dim_reviews = (df_dim_reviews
    .withColumn("sk_review_id", F.row_number().over(janela_sk_reviews).cast("bigint"))
)

#selecao de colunas para gravacao
df_dim_reviews = df_dim_reviews.select(
    "sk_review_id",
    "sk_movie_id",
    "qtd_avaliacoes_usuarios",
    "nota_media_usuarios"
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_reviews")

display(spark.table("gold.dim_reviews").limit(1000))

In [0]:
#aqui criaremos 3 tabelas bridge ligando 
    #1. movie -> genre
    #2. movie -> person
    #3. movie -> company

from pyspark.sql.window import Window
import pyspark.sql.functions as F

#1. movie -> genre
#carregando a tabela silver de generos e as dimensoes respectivas
df_silver_generos = spark.table("silver.tb_generos")
df_dim_movies = spark.table("gold.dim_movies")
df_dim_genres = spark.table("gold.dim_genres")

df_bridge_genre = (df_silver_generos
    .join(df_dim_movies, df_silver_generos["id_filme"] == df_dim_movies["id_filme"], "inner")
    .join(df_dim_genres, df_silver_generos["nome_genero"] == df_dim_genres["nome_genero"], "inner")
    .select(
        col("sk_movie_id").cast("bigint"),
        col("sk_genre_id").cast("bigint")
    )
    .dropDuplicates()
)

df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_genre")


#2. movie -> person
#carregando a tabela silver de entidades e as dimensoes respectivas
df_silver_entidades = spark.table("silver.tb_pessoas_empresas")
df_dim_people = spark.table("gold.dim_people")

#filtramos apenas as pessoas na silver
df_silver_people_only = df_silver_entidades.filter(col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))

df_bridge_person = (df_silver_people_only
    .join(df_dim_movies, df_silver_people_only["id_filme"] == df_dim_movies["id_filme"], "inner")
    .join(
        df_dim_people, 
        (df_silver_people_only["nome_entidade"] == df_dim_people["nome_pessoa"]) & 
        (df_silver_people_only["tipo_entidade"] == df_dim_people["tipo_pessoa"]), 
        "inner"
    )
    .select(
        col("sk_movie_id").cast("bigint"),
        col("sk_person_id").cast("bigint")
    )
    .dropDuplicates()
)

df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_person")

#3. movie -> company
df_dim_companies = spark.table("gold.dim_companies")

#filtramos apenas as produtoras na silver
df_silver_companies_only = df_silver_entidades.filter(col("tipo_entidade") == "Produtora")

df_bridge_company = (df_silver_companies_only
    .join(df_dim_movies, df_silver_companies_only["id_filme"] == df_dim_movies["id_filme"], "inner")
    .join(
        df_dim_companies, 
        df_silver_companies_only["nome_entidade"] == df_dim_companies["nome_empresa"], 
        "inner"
    )
    .select(
        col("sk_movie_id").cast("bigint"),
        col("sk_company_id").cast("bigint")
    )
    .dropDuplicates()
)

df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.bridge_movie_company")

print("--- Tabela: gold.bridge_movie_genre ---")
display(spark.table("gold.bridge_movie_genre").limit(5))

print("--- Tabela: gold.bridge_movie_person ---")
display(spark.table("gold.bridge_movie_person").limit(5))

print("--- Tabela: gold.bridge_movie_company ---")
display(spark.table("gold.bridge_movie_company").limit(5))

# ENTREGA 2
- Tabela de contexto para uso do time de GenAI

In [0]:
import pyspark.sql.functions as F

#importacao das tabelas e dimensoes
df_dim_movies = spark.table("gold.dim_movies")
df_fact = spark.table("gold.fact_movies_performance")
df_bridge_person = spark.table("gold.bridge_movie_person")
df_dim_people = spark.table("gold.dim_people")

#agregação de atores 
df_atores = (df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores"))
)

#agregação de diretores
df_diretores = (df_bridge_person
    .join(df_dim_people, on="sk_person_id", how="inner")
    .filter(F.col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("diretores"))
)

#junção de todos os dados de pessoas e filmes num unico df
df_base = (df_dim_movies
    .join(df_fact, on="sk_movie_id", how="left")
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
)

#=======================================================================================================================
#TRATAMENTO DE NULOS, usando coalesce para substituição contextual
#substituindo valores NULL por "desconhecido"
df_context = df_base.withColumn("titulo_safe", F.coalesce(F.col("titulo"), F.lit("Título Desconhecido")))
df_context = df_context.withColumn("ano_safe", F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("desconhecido")))

#substituindo valores financeiros, adicionando "US$" se exister, ou o texto de segurança caso seja nulo
df_context = df_context.withColumn(
    "receita_safe", 
    F.when(F.col("receita_usd").isNotNull(), F.concat(F.lit("US$ "), F.col("receita_usd").cast("string")))
     .otherwise(F.lit("um valor não divulgado"))
)
df_context = df_context.withColumn(
    "orcamento_safe", 
    F.when(F.col("orcamento_usd").isNotNull(), F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string")))
     .otherwise(F.lit("um valor não divulgado"))
)

df_context = df_context.withColumn("atores_safe", F.coalesce(F.col("atores"), F.lit("elenco não informado")))
df_context = df_context.withColumn("diretores_safe", F.coalesce(F.col("diretores"), F.lit("um diretor não informado")))
df_context = df_context.withColumn("sinopse_safe", F.coalesce(F.col("sinopse"), F.lit("Sinopse indisponível.")))
#=======================================================================================================================

#concatenacao de contexto para o modelo de linguagem llm
df_context = df_context.withColumn(
    "llm_context_document",
    F.concat(
        F.lit("O filme "), F.col("titulo_safe"),
        F.lit(", lançado no ano de "), F.col("ano_safe"),
        F.lit(", faturou "), F.col("receita_safe"),
        F.lit(" e teve um custo de "), F.col("orcamento_safe"),
        F.lit(". Estrelado por "), F.col("atores_safe"),
        F.lit(" e dirigido por "), F.col("diretores_safe"),
        F.lit(", o filme possui a seguinte sinopse: "), F.col("sinopse_safe")
    )
)

df_genai = df_context.select(
    F.col("id_filme").alias("movie_id"),
    F.col("titulo_safe").alias("title"),
    "llm_context_document"
)


df_genai.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_genai_movies_context")

#visualização dos testes 
pd_display = spark.table("gold.gold_genai_movies_context").limit(10).toPandas()
for index, row in pd_display.iterrows():
    print(f"\n--- Filme: {row['title']} ---")
    print(row['llm_context_document'])
